## Award modeling

Predicting Oscar outcomes for nominated films. Inputs come from the
team pipeline in `data/processed/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PROCESSED = Path("../data/processed")

oscars = pd.read_csv(DATA_PROCESSED / "oscars_cleaned.csv")
imdb = pd.read_csv(DATA_PROCESSED / "imdb_cleaned.csv", dtype={"tconst": "str"})

print(f"oscars: {oscars.shape}")
print(f"imdb:   {imdb.shape}")


oscars: (12137, 11)
imdb:   (709446, 9)


### TMDB review features

Per-film review aggregates from our TMDB pull, joined through `imdb_id`.

In [2]:
movies = pd.read_csv("../data/raw/tmdb_movies.csv", usecols=["tmdb_id", "imdb_id"])
review_features = pd.read_csv(DATA_PROCESSED / "review_features_by_movie.csv")

bridge = movies.dropna(subset=["imdb_id"]).drop_duplicates("imdb_id")[["imdb_id", "tmdb_id"]]
review_by_imdb = bridge.merge(review_features, on="tmdb_id", how="inner")

film_ids = oscars["FilmId"].dropna().str.split("|").explode().str.strip()
covered = film_ids.isin(set(review_by_imdb["imdb_id"]))

print(f"review features: {review_by_imdb.shape}")
print(f"oscar film rows with review features: {covered.sum()} of {len(film_ids)} ({covered.mean() * 100:.1f}%)")


review features: (11508, 29)
oscar film rows with review features: 6101 of 10878 (56.1%)


### Analysis frame: 1950-2025

Everything below is restricted to ceremonies from 1950 onward. The candidate universe is built in the next section from `imdb_cleaned` (one row per film) joined to the TMDB pull, and the modelling pool keeps only films with a TMDB row - the pull's `vote_count >= 100` floor - because that is the population the review data can describe.

In [3]:
FIRST_YEAR = 1950

oscars = oscars[oscars["Year"] >= FIRST_YEAR].copy()
print(f"ceremonies {oscars.Year.min()}-{oscars.Year.max()} | nomination rows {len(oscars):,}")


ceremonies 1950-2025 | nomination rows 9,629


### Target: Best Picture nomination

Label = the film was nominated for Best Picture, win or lose. 451 films across 76 ceremonies.

The first model drops rows with missing predictors and keeps only films with a TMDB row: the pull's `vote_count >= 100` floor already excludes obscure films, which is the population we want to score. TMDB budget and revenue record unknown as 0, so those zeros become NaN and those rows drop with the rest of the missing values.

In [4]:
bp = oscars[(oscars["CanonicalCategory"] == "BEST PICTURE") & oscars["FilmId"].notna()]
bp = bp[~bp["FilmId"].str.contains(r"\|")]
nominated_ids = set(bp["FilmId"].str.strip())

films = imdb[imdb["startYear"].between(1950, 2025)].copy()
films["startYear"] = films["startYear"].astype(int)
films["nominated"] = films["tconst"].isin(nominated_ids).astype(int)

model_rows = films.dropna(subset=["runtimeMinutes", "averageRating", "numVotes", "genres"]).copy()
print(f"complete-case IMDb rows: {len(model_rows):,} | nominees {int(model_rows.nominated.sum())} | base rate {model_rows.nominated.mean() * 100:.3f}%")

tmdb_fields = ["imdb_id", "popularity", "cast_count", "crew_count", "n_directors",
               "vfx_crew", "review_total", "budget", "revenue", "vote_average", "vote_count"]
tmdb = pd.read_csv("../data/raw/tmdb_movies.csv", usecols=tmdb_fields)
tmdb[["budget", "revenue"]] = tmdb[["budget", "revenue"]].replace(0, np.nan)
tmdb = tmdb.dropna(subset=["imdb_id"])

pool = model_rows.merge(tmdb, left_on="tconst", right_on="imdb_id", how="inner", validate="m:1")
print(f"pool with a TMDB row:   {len(pool):,} | nominees {int(pool.nominated.sum())} | base rate {pool.nominated.mean() * 100:.2f}%")
print(pool[["popularity", "cast_count", "crew_count", "n_directors", "vfx_crew",
             "review_total", "budget", "revenue"]].isna().sum().to_string())

pool = pool.merge(review_by_imdb.drop(columns=["title"]), on="imdb_id", how="left")
print(f"with review features:   {int(pool['reviews'].notna().sum()):,} of {len(pool):,} rows | nominees {int(pool.loc[pool['reviews'].notna(), 'nominated'].sum())}")


complete-case IMDb rows: 287,076 | nominees 451 | base rate 0.157%
pool with a TMDB row:   20,800 | nominees 433 | base rate 2.08%
popularity          0
cast_count          0
crew_count          0
n_directors         0
vfx_crew            0
review_total        0
budget          10250
revenue          8532
with review features:   10,347 of 20,800 rows | nominees 383


### Imputation

Two mechanisms, because the missingness has two causes.

**Structural (no imputation needed).** A film with `review_total == 0` has no reviews, so its mention counts are 0 by construction - 12,169 of the 23,677 pulled films. Those cells are filled with 0 and a `has_reviews` flag records the difference. `reviews` is dropped: it equals `review_total` on all 11,508 films that have reviews, so it carries no information of its own.

**Regression with noise**, as in the course notes: fit an OLS on the rows where the column is observed, predict the missing rows, add Gaussian noise with sd equal to the imputer's residual sd (seeded, repeatable). Applied to `budget`, `revenue`, `mean_rating` and the twelve sentiment scores - including for films with no reviews, where the score itself is undefined. Predictions are clipped to each column's observed range.

In [5]:
import statsmodels.api as sm

REVIEW_COUNTS = ['positivity_n', 'negativity_n', 'performance_mentions_n', 'performance_tone_n', 'production_mentions_n', 'production_tone_n', 'effects_mentions_n', 'effects_tone_n', 'humor_mentions_n', 'humor_tone_n', 'aesthetics_mentions_n', 'aesthetics_tone_n']
REVIEW_SCORES = ['positivity', 'negativity', 'performance_mentions', 'performance_tone', 'production_mentions', 'production_tone', 'effects_mentions', 'effects_tone', 'humor_mentions', 'humor_tone', 'aesthetics_mentions', 'aesthetics_tone']

pool_raw = pool.copy()   # untouched copy for the pre-production model
pool["has_reviews"] = (pool["review_total"] > 0).astype(int)
pool[REVIEW_COUNTS] = pool[REVIEW_COUNTS].fillna(0)
pool = pool.drop(columns=["reviews"])
print(f"films with no reviews (counts set to 0): {int((pool.has_reviews == 0).sum()):,}")

TO_IMPUTE = ["budget", "revenue", "mean_rating"] + REVIEW_SCORES
CANDIDATES = ["averageRating", "numVotes", "runtimeMinutes", "popularity", "cast_count",
              "crew_count", "n_directors", "vfx_crew", "review_total", "has_reviews"] + \
             REVIEW_COUNTS + TO_IMPUTE

rng = np.random.default_rng(42)
log = []
for column in TO_IMPUTE:
    missing = pool[column].isna()
    if missing.sum() == 0:
        continue
    predictors = [c for c in CANDIDATES if c != column and pool[c].notna().all()]
    observed = pool.loc[~missing]
    mean_before = pool[column].mean()
    fit = sm.OLS(observed[column], sm.add_constant(observed[predictors], has_constant="add")).fit()
    predicted = fit.predict(sm.add_constant(pool.loc[missing, predictors], has_constant="add"))
    noise = rng.normal(0, fit.resid.std(ddof=1), size=int(missing.sum()))
    values = (predicted + noise).clip(pool[column].min(), pool[column].max())
    pool[column] = pool[column].where(~missing, values)
    log.append({"column": column, "imputed": int(missing.sum()), "R2": fit.rsquared,
                "resid_sd": fit.resid.std(ddof=1),
                "mean_before": mean_before, "mean_after": pool[column].mean()})

imputation_log = pd.DataFrame(log)
print(f"rows: {len(pool):,} | nominees {int(pool.nominated.sum())} | base rate {pool.nominated.mean() * 100:.2f}% | remaining NaN: {int(pool[TO_IMPUTE + REVIEW_COUNTS].isna().sum().sum())}")
imputation_log.round(3)


films with no reviews (counts set to 0): 10,453
rows: 20,800 | nominees 433 | base rate 2.08% | remaining NaN: 0


,column,imputed,R2,resid_sd,mean_before,mean_after
0,budget,10250,0.495,2.860085e+07,2.624934e+07,2.205176e+07
1,revenue,8532,0.616,9.369392e+07,6.234251e+07,5.645525e+07
2,mean_rating,10729,0.326,1.385000e+00,6.458000e+00,6.059000e+00
3,positivity,10895,0.149,5.900000e-02,5.750000e-01,4.310000e-01
4,negativity,13151,0.187,6.500000e-02,5.560000e-01,3.900000e-01
5,performance_mentions,13406,0.270,1.260000e-01,1.880000e-01,1.380000e-01
6,performance_tone,16276,0.394,6.500000e-02,5.470000e-01,4.230000e-01
7,production_mentions,14028,0.221,1.270000e-01,1.970000e-01,1.510000e-01
8,production_tone,17796,0.348,6.700000e-02,5.460000e-01,4.160000e-01
9,effects_mentions,18030,0.265,1.160000e-01,1.380000e-01,1.000000e-01


### Model 1: logistic regression

Predictors: IMDb rating/votes/runtime, TMDB popularity/credits/review count/budget/revenue, review volume (`mean_rating`, `has_reviews`), the twelve mention counts, the twelve sentiment scores and genre indicators. L2 logistic at sklearn's default C = 1, fit on the training split only - the holdout is scored once, in the final section, by the model chosen through CV.

No temporal split - a random stratified 80/20 split is fine for this project.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

NUMERIC = ["averageRating", "numVotes", "runtimeMinutes", "popularity", "cast_count",
           "crew_count", "n_directors", "vfx_crew", "review_total", "budget", "revenue"]
REVIEW = ["mean_rating", "has_reviews"] + REVIEW_COUNTS + REVIEW_SCORES

genre_flags = pool["genres"].str.get_dummies(sep=",")
top_genres = genre_flags.sum().sort_values(ascending=False).head(12).index.tolist()
pool[top_genres] = genre_flags[top_genres].astype(int)

data = pool.dropna(subset=NUMERIC + REVIEW).copy()
print(f"rows: {len(data):,} | nominees {int(data.nominated.sum())} | base rate {data.nominated.mean() * 100:.2f}%")

features = NUMERIC + REVIEW + top_genres
X = data[features]
y = data["nominated"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(f"train {len(X_train):,} ({int(y_train.sum())} nominees) | test {len(X_test):,} ({int(y_test.sum())} nominees)")

def make_model(C=1.0, l1_ratio=0.0, solver="lbfgs", max_iter=5000):
    return Pipeline(steps=[
        ("preprocess", ColumnTransformer(transformers=[
            ("num", StandardScaler(), NUMERIC + REVIEW),
            ("bin", "passthrough", top_genres),
        ])),
        ("model", LogisticRegression(C=C, l1_ratio=l1_ratio, solver=solver, max_iter=max_iter)),
    ])

model = make_model()
model.fit(X_train, y_train)

coefs = pd.Series(model.named_steps["model"].coef_[0], index=features)
coef_table = pd.DataFrame({"coefficient": coefs, "odds_ratio": np.exp(coefs)})
coef_table.reindex(coef_table["coefficient"].abs().sort_values(ascending=False).index).round(3)


rows: 20,800 | nominees 433 | base rate 2.08%
train 16,640 (346 nominees) | test 4,160 (87 nominees)


,coefficient,odds_ratio
averageRating,1.589,4.899
Action,-1.479,0.228
Horror,-1.340,0.262
Drama,1.329,3.778
Sci-Fi,-0.847,0.429
Crime,-0.670,0.512
Biography,0.537,1.710
Fantasy,-0.535,0.586
Romance,0.503,1.653
budget,-0.473,0.623


### Penalty selection: ridge, 10-fold CV

L2 logistic (`lbfgs`, `l1_ratio=0`) over the C grid, scored by mean CV log loss across 10 stratified folds. The base-rate log loss is the reference a model has to beat.

In [7]:
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, cross_validate

C_values = [1/32, 1/16, 1/8, 1/4, 1/2, 1, 2, 4, 8, 16, 32]
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

base_ll = log_loss(y, np.full(len(y), y.mean()))
print(f"rows {len(X):,} | predictors {len(features)} | base-rate log loss {base_ll:.6f}")

rows = []
for C in C_values:
    res = cross_validate(make_model(C), X, y, cv=cv, scoring=["neg_log_loss", "roc_auc"], n_jobs=-1)
    rows.append({"C": C,
                 "log_loss": -res["test_neg_log_loss"].mean(),
                 "sd": res["test_neg_log_loss"].std(),
                 "AUC": res["test_roc_auc"].mean()})

results = pd.DataFrame(rows)
best_C = results.loc[results["log_loss"].idxmin(), "C"]
print(f"best C = {best_C} at mean CV log loss {results['log_loss'].min():.6f}")
results.round(6)


rows 20,800 | predictors 49 | base-rate log loss 0.101203
best C = 0.25 at mean CV log loss 0.057650


,C,log_loss,sd,AUC
0,0.03125,0.058604,0.003005,0.953672
1,0.06250,0.057920,0.003260,0.953764
2,0.12500,0.057668,0.003485,0.953288
3,0.25000,0.057650,0.003667,0.952727
4,0.50000,0.057672,0.003773,0.952263
5,1.00000,0.057836,0.003850,0.951759
6,2.00000,0.057887,0.003881,0.951464
7,4.00000,0.057911,0.003902,0.951354
8,8.00000,0.057975,0.003912,0.951169
9,16.00000,0.057972,0.003899,0.951138


### Selected penalty: coefficients and holdout performance

C is the grid value with the lowest mean CV log loss, and the CV row shown is that winning row only.

The selected model is then fit on **`X_train` only** and scored once on the original holdout from the first model section (`X_test`, 20% of the rows, same seed). Fitting on all rows and then scoring the holdout would leak those rows back into the estimate, so the coefficients below are also the train-fit ones.

In [8]:
from sklearn.metrics import (accuracy_score, confusion_matrix, log_loss,
                             precision_score, recall_score, roc_auc_score)

best = results.loc[results["log_loss"].idxmin()]
C_best = best["C"]

print(f"best C = {C_best} | mean CV log loss {best['log_loss']:.6f} | "
      f"fold sd {best['sd']:.6f} | AUC {best['AUC']:.4f}")
display(pd.DataFrame([best]).round(6))

final_model = make_model(C_best).fit(X_train, y_train)
p_test = final_model.predict_proba(X_test)[:, 1]
y_hat = (p_test >= 0.5).astype(int)
base_rate = y_train.mean()

print(f"\nholdout: {len(X_test):,} rows, {int(y_test.sum())} nominees")
print(f"  log loss   {log_loss(y_test, p_test):.6f}   (base rate {log_loss(y_test, np.full(len(y_test), base_rate)):.6f})")
print(f"  AUC        {roc_auc_score(y_test, p_test):.4f}")
print(f"  accuracy   {accuracy_score(y_test, y_hat):.3f}")
print(f"  precision  {precision_score(y_test, y_hat):.3f}")
print(f"  recall     {recall_score(y_test, y_hat):.3f}")
print(confusion_matrix(y_test, y_hat))

coefs = pd.Series(final_model.named_steps["model"].coef_[0], index=features)
odds = pd.DataFrame({"coefficient": coefs, "odds_ratio": np.exp(coefs)})
odds.reindex(odds["coefficient"].abs().sort_values(ascending=False).index).round(3)


best C = 0.25 | mean CV log loss 0.057650 | fold sd 0.003667 | AUC 0.9527


,C,log_loss,sd,AUC
3,0.25,0.05765,0.003667,0.952727



holdout: 4,160 rows, 87 nominees
  log loss   0.053771   (base rate 0.101574)
  AUC        0.9628
  accuracy   0.981
  precision  0.625
  recall     0.287
[[4058   15]
 [  62   25]]


,coefficient,odds_ratio
averageRating,1.522,4.581
Drama,1.218,3.381
Action,-1.147,0.318
Horror,-0.867,0.420
Crime,-0.603,0.547
Sci-Fi,-0.584,0.558
Biography,0.538,1.712
Romance,0.490,1.633
budget,-0.453,0.635
has_reviews,0.401,1.493


## Model 2: pre-production predictors only

Same model, same folds, same C grid - but restricted to what a producer could know before shooting: `budget`, `cast_count`, `crew_count`, `n_directors`, `vfx_crew`, `runtimeMinutes` and the genre indicators.

Everything downstream of release is removed: IMDb rating and votes, TMDB popularity and review count, revenue, and the whole review block. The imputation is redone here for the same reason - the full model's `budget` was imputed from post-release columns, which would smuggle future information into an ex-ante model. Here budget is imputed from pre-production predictors only.

Caveat to carry into the write-up: the candidate pool is still films with 100+ TMDB votes, which is itself a fact about later reception. Features are ex-ante; the universe is not.

In [9]:
PRE_NUMERIC = ["budget", "cast_count", "crew_count", "n_directors", "vfx_crew", "runtimeMinutes"]

exante = pool_raw[["tconst", "startYear", "nominated"] + PRE_NUMERIC + ["genres"]].copy()
pre_genre_flags = exante["genres"].str.get_dummies(sep=",")
pre_genres = pre_genre_flags.sum().sort_values(ascending=False).head(12).index.tolist()
exante[pre_genres] = pre_genre_flags[pre_genres].astype(int)

print(f"rows: {len(exante):,} | nominees {int(exante.nominated.sum())} | base rate {exante.nominated.mean() * 100:.2f}%")
print(f"missing before imputation: {exante[PRE_NUMERIC].isna().sum().to_dict()}")

rng = np.random.default_rng(42)
for column in PRE_NUMERIC:
    missing = exante[column].isna()
    if missing.sum() == 0:
        continue
    predictors = [c for c in PRE_NUMERIC + pre_genres
                  if c != column and exante[c].notna().all()]
    observed = exante.loc[~missing]
    fit = sm.OLS(observed[column],
                 sm.add_constant(observed[predictors], has_constant="add")).fit()
    predicted = fit.predict(sm.add_constant(exante.loc[missing, predictors], has_constant="add"))
    noise = rng.normal(0, fit.resid.std(ddof=1), size=int(missing.sum()))
    exante[column] = exante[column].where(
        ~missing, (predicted + noise).clip(exante[column].min(), exante[column].max()))
    print(f"{column}: imputed {int(missing.sum()):,} rows, R2 {fit.rsquared:.3f}, from {len(predictors)} pre-production predictors")

print(f"remaining NaN: {int(exante[PRE_NUMERIC].isna().sum().sum())}")


rows: 20,800 | nominees 433 | base rate 2.08%
missing before imputation: {'budget': 10250, 'cast_count': 0, 'crew_count': 0, 'n_directors': 0, 'vfx_crew': 0, 'runtimeMinutes': 0}
budget: imputed 10,250 rows, R2 0.453, from 17 pre-production predictors
remaining NaN: 0


### Model 2: penalty selection, 10-fold CV

Identical protocol to Model 1: the same `C_values`, the same 10 folds, the same scoring.

In [10]:
def make_pre_pipeline(C):
    return Pipeline(steps=[
        ("preprocess", ColumnTransformer(transformers=[
            ("num", StandardScaler(), PRE_NUMERIC),
            ("bin", "passthrough", pre_genres),
        ])),
        ("model", LogisticRegression(C=C, l1_ratio=0, solver="lbfgs", max_iter=5000)),
    ])

pre_features = PRE_NUMERIC + pre_genres
X_pre = exante[pre_features]
y_pre = exante["nominated"]

pre_base_ll = log_loss(y_pre, np.full(len(y_pre), y_pre.mean()))
print(f"rows {len(X_pre):,} | predictors {len(pre_features)} | base-rate log loss {pre_base_ll:.6f}")

pre_rows = []
for C in C_values:
    res = cross_validate(make_pre_pipeline(C), X_pre, y_pre, cv=cv,
                         scoring=["neg_log_loss", "roc_auc"], n_jobs=-1)
    pre_rows.append({"C": C,
                     "log_loss": -res["test_neg_log_loss"].mean(),
                     "sd": res["test_neg_log_loss"].std(),
                     "AUC": res["test_roc_auc"].mean()})
pre_results = pd.DataFrame(pre_rows)
best_pre_C = pre_results.loc[pre_results["log_loss"].idxmin(), "C"]
print(f"best C = {best_pre_C} at mean CV log loss {pre_results['log_loss'].min():.6f}")
pre_results.round(6)


rows 20,800 | predictors 18 | base-rate log loss 0.101203


best C = 1.0 at mean CV log loss 0.077318


,C,log_loss,sd,AUC
0,0.03125,0.078222,0.002925,0.889400
1,0.06250,0.077666,0.003039,0.888692
2,0.12500,0.077419,0.003120,0.887621
3,0.25000,0.077330,0.003173,0.886725
4,0.50000,0.077319,0.003205,0.885921
5,1.00000,0.077318,0.003200,0.885504
6,2.00000,0.077348,0.003219,0.884971
7,4.00000,0.077373,0.003236,0.884766
8,8.00000,0.077364,0.003239,0.884683
9,16.00000,0.077372,0.003242,0.884637


### Model 2: selected penalty, coefficients and holdout performance

Same seed and row order as Model 1, so the holdout rows are the same 20%.

In [11]:
from sklearn.metrics import (accuracy_score, confusion_matrix, log_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split

best_pre = pre_results.loc[pre_results["log_loss"].idxmin()]
C_pre = best_pre["C"]

print(f"best C = {C_pre} | mean CV log loss {best_pre['log_loss']:.6f} | "
      f"fold sd {best_pre['sd']:.6f} | AUC {best_pre['AUC']:.4f}")
display(pd.DataFrame([best_pre]).round(6))

X_pre_train, X_pre_test, y_pre_train, y_pre_test = train_test_split(
    X_pre, y_pre, test_size=0.2, stratify=y_pre, random_state=42)
final_pre = make_pre_pipeline(C_pre).fit(X_pre_train, y_pre_train)
p_pre_test = final_pre.predict_proba(X_pre_test)[:, 1]
y_hat_pre = (p_pre_test >= 0.5).astype(int)
pre_base_rate = y_pre_train.mean()

print(f"\nholdout: {len(X_pre_test):,} rows, {int(y_pre_test.sum())} nominees")
print(f"  log loss   {log_loss(y_pre_test, p_pre_test):.6f}   (base rate {log_loss(y_pre_test, np.full(len(y_pre_test), pre_base_rate)):.6f})")
print(f"  AUC        {roc_auc_score(y_pre_test, p_pre_test):.4f}")
print(f"  accuracy   {accuracy_score(y_pre_test, y_hat_pre):.3f}")
print(f"  precision  {precision_score(y_pre_test, y_hat_pre):.3f}")
print(f"  recall     {recall_score(y_pre_test, y_hat_pre):.3f}")
print(confusion_matrix(y_pre_test, y_hat_pre))

pre_coefs = pd.Series(final_pre.named_steps["model"].coef_[0], index=pre_features)
pre_odds = pd.DataFrame({"coefficient": pre_coefs, "odds_ratio": np.exp(pre_coefs)})
pre_odds.reindex(pre_odds["coefficient"].abs().sort_values(ascending=False).index).round(3)


best C = 1.0 | mean CV log loss 0.077318 | fold sd 0.003200 | AUC 0.8855


,C,log_loss,sd,AUC
5,1.0,0.077318,0.0032,0.885504



holdout: 4,160 rows, 87 nominees
  log loss   0.073646   (base rate 0.101574)
  AUC        0.9022
  accuracy   0.979
  precision  0.529
  recall     0.103
[[4065    8]
 [  78    9]]


,coefficient,odds_ratio
Action,-1.633,0.195
Drama,1.317,3.732
Horror,-1.098,0.333
crew_count,0.565,1.760
runtimeMinutes,0.483,1.621
Crime,-0.463,0.629
Comedy,-0.374,0.688
Sci-Fi,-0.369,0.692
Thriller,-0.347,0.707
Biography,0.339,1.403
